## Step 1: Project Initialization and Dependency Setup

In [ ]:
!pip install groq chromadb sentence-transformers -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2: Data Loading and Preprocessing

**Fix 1 (Label alignment):** We now parse `anomaly_types`, `latency`, and `block_id` directly
from the RAG document text — these are the fields generated by LAD4. This ensures the metadata
stored in ChromaDB comes from the actual document, not from a separate join that could mismatch.

In [ ]:
import pandas as pd
import re

rag_df = pd.read_csv("/content/drive/MyDrive/Log_Anamoly_Detection/results/rag_documents.csv")
documents = rag_df["RAG_Document"].tolist()

print("Total RAG documents:", len(documents))
print("\nSample document:")
print(documents[0])

In [ ]:
# ── FIX 1 & 3: Parse structured metadata from each RAG document ──────────────
# Instead of relying on a separate CSV join, we extract all metadata fields
# directly from the RAG document text. This means block_id, anomaly_types and
# latency stored in ChromaDB come from the same source as the document content.

def parse_metadata_from_doc(doc: str) -> dict:
    """
    Extract structured metadata fields from a RAG document string.
    Returns a dict with block_id, anomaly_types (primary), latency, total_events.
    """
    block_id_match  = re.search(r'Block ID\s*:\s*(\S+)', doc)
    label_match     = re.search(r'^Label\s*:\s*(.+)', doc, re.MULTILINE)
    atypes_match    = re.search(r'Anomaly Type\(s\)\s*:\s*(.+)', doc)
    latency_match   = re.search(r'Latency\s*:\s*(\d+)', doc)
    total_ev_match  = re.search(r'Total Events\s*:\s*(\d+)', doc)

    full_types = atypes_match.group(1).strip() if atypes_match else 'unknown'
    # Primary type = first type listed (used for ChromaDB metadata filter)
    primary_type = full_types.split(',')[0].strip()

    return {
        "block_id":      block_id_match.group(1).strip() if block_id_match else 'unknown',
        "label":         label_match.group(1).strip()    if label_match    else 'Fail',
        "anomaly_types": full_types,
        "primary_type":  primary_type,
        "latency":       int(latency_match.group(1))     if latency_match  else 0,
        "total_events":  int(total_ev_match.group(1))    if total_ev_match else 0,
    }

# Parse metadata for all documents
all_metadata = [parse_metadata_from_doc(doc) for doc in documents]

# Show distribution of anomaly types
import pandas as pd
meta_df = pd.DataFrame(all_metadata)
print("Anomaly type distribution:")
print(meta_df['primary_type'].value_counts().to_string())
print("\nSample parsed metadata:")
print(all_metadata[0])

## Step 3: Semantic Embedding Generation

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings  = embed_model.encode(documents, show_progress_bar=True)

print("Embedding shape:", embeddings.shape)

# FIX: Save embeddings so main.py (Render deployment) can load them at startup.
# Without this step, main.py raises FileNotFoundError on embeddings.npy.
EMBEDDINGS_SAVE_PATH = "/content/drive/MyDrive/Log_Anamoly_Detection/results/embeddings.npy"
np.save(EMBEDDINGS_SAVE_PATH, embeddings)
print(f"Embeddings saved to {EMBEDDINGS_SAVE_PATH}")


## Step 4: Vector Database Indexing with ChromaDB

In [ ]:
import chromadb

# FIX: chromadb.Client(Settings(...)) is deprecated.
# Use PersistentClient for disk-backed storage (saves across Colab sessions).
chroma_client = chromadb.PersistentClient(path="./chroma_db")

try:
    chroma_client.delete_collection(name="anomaly_logs")
    print("Old collection deleted — rebuilding with metadata.")
except Exception:
    pass

collection = chroma_client.get_or_create_collection(name="anomaly_logs")

BATCH_SIZE = 500

for start in range(0, len(documents), BATCH_SIZE):
    end = min(start + BATCH_SIZE, len(documents))
    collection.add(
        documents  = documents[start:end],
        embeddings = embeddings[start:end].tolist(),
        ids        = [str(i) for i in range(start, end)],
        metadatas  = all_metadata[start:end],
    )
    print(f"  Indexed {end}/{len(documents)} documents...")

print(f"\nTotal vectors in ChromaDB: {collection.count()}")


## Step 5: Retrieval Logic

In [ ]:
def retrieve_similar_anomalies(
    query: str,
    k: int = 3,
    anomaly_type_filter: str = None
) -> list[dict]:
    """
    Retrieve k most semantically similar anomaly documents for a query.

    Args:
        query:               Free-text query or auto-constructed summary string.
        k:                   Number of results to retrieve.
        anomaly_type_filter: Optional primary_type to restrict search scope
                             (e.g. 'high_latency', 'repetition', 'missing_events',
                              'duplicate_pattern').

    Returns:
        List of dicts with keys: document, block_id, primary_type, distance.
    """
    query_embedding = embed_model.encode([query]).tolist()

    # ── FIX 4: Apply metadata filter when anomaly type is known ───────────────
    where_clause = None
    if anomaly_type_filter:
        where_clause = {"primary_type": {"$eq": anomaly_type_filter}}

    results = collection.query(
        query_embeddings = query_embedding,
        n_results        = k,
        where            = where_clause,
    )

    retrieved_docs = []
    for doc, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
    ):
        retrieved_docs.append({
            "document":     doc,
            "block_id":     meta.get("block_id", "unknown"),   # FIX 3
            "primary_type": meta.get("primary_type", "unknown"),
            "latency":      meta.get("latency", 0),
            "distance":     dist,
        })

    return retrieved_docs

## Step 6: LLM Root Cause Analysis

In [ ]:
from groq import Groq

groq_client = Groq(api_key="your api key")  # <-- replace with your key

In [ ]:
import json
import re
from collections import Counter

# ── EVENT DESCRIPTIONS ────────────────────────────────────────────────────────
EVENT_DESCRIPTIONS = {
    "E1":  "A DataNode received a request to store a block that already exists — duplicate write or stale reference.",
    "E2":  "Block checksum verification passed — data written is confirmed intact.",
    "E3":  "A DataNode successfully served a block read request.",
    "E4":  "An exception occurred while serving a block — read or transfer failed.",
    "E5":  "A DataNode started receiving a new block — start of a pipeline write.",
    "E6":  "A DataNode finished receiving a full block — complete transfer confirmed.",
    "E7":  "An exception was thrown during a block write operation.",
    "E8":  "The PacketResponder thread was interrupted — unexpected pipeline disruption.",
    "E9":  "A DataNode finished receiving a block of a known size — successful replica receipt.",
    "E10": "The PacketResponder thread threw an unhandled exception.",
    "E11": "The PacketResponder thread for a block terminated — normal end or failure.",
    "E12": "Exception while writing block to a mirror DataNode — replication pipeline failed.",
    "E13": "DataNode received an empty packet — heartbeat or end-of-stream signal.",
    "E14": "Exception inside receiveBlock handler — block write could not complete.",
    "E15": "NameNode adjusted block offset metadata — recovery or corruption repair.",
    "E16": "Block successfully transferred to another DataNode.",
    "E17": "Block transfer to target DataNode failed — re-replication unsuccessful.",
    "E18": "NameNode instructed DataNode to start background block copy thread.",
    "E19": "Block file reopened for appending.",
    "E20": "Delete failed — block metadata not found in DataNode volume map (orphaned block).",
    "E21": "DataNode deleted a block file from local disk — triggered by NameNode invalidation.",
    "E22": "NameNode allocated a new block ID — start of new block creation.",
    "E23": "NameNode added block to invalidation set — scheduled for deletion.",
    "E24": "NameNode removed block from replication queue — no longer belongs to any file.",
    "E25": "NameNode instructed DataNode to replicate block — under-replication detected.",
    "E26": "NameNode updated block map after DataNode reported successful storage.",
    "E27": "Redundant block storage report received — duplicate reporting.",
    "E28": "Block report for unknown file received — block is orphaned.",
    "E29": "Replication request timed out — target DataNode did not complete copy in time.",
}

VALID_ANOMALY_TYPES = {"duplicate_pattern", "repetition", "missing_events", "high_latency"}

FALLBACK_MITIGATIONS = {
    "duplicate_pattern": {
        "high":   ["Check DataNode pipeline threads for race conditions causing duplicate block writes.",
                   "Enable idempotency checks on NameNode to reject duplicate block registrations."],
        "medium": ["Review client retry config — reduce max retries or add backoff jitter.",
                   "Monitor DataNode network throughput for intermittent failures."],
        "low":    ["Audit HDFS client version for known duplicate-write bugs."],
    },
    "repetition": {
        "high":   ["Inspect DataNode for stuck PacketResponder thread and restart if confirmed.",
                   "Check for network packet loss between pipeline DataNodes."],
        "medium": ["Review NameNode RPC logs for timeout patterns triggering re-sends.",
                   "Verify DataNode JVM heap — GC pauses can stall pipeline and trigger retries."],
        "low":    ["Increase pipeline write timeout thresholds to reduce false retry triggers."],
    },
    "missing_events": {
        "high":   ["Inspect DataNode that aborted pipeline — check for disk errors or OOM.",
                   "Verify block replication in NameNode — trigger re-replication if under-replicated."],
        "medium": ["Review DataNode stderr logs around block timestamp for crash evidence.",
                   "Check network stability between pipeline nodes for partial disconnects."],
        "low":    ["Add HDFS block scanner runs to detect and repair corrupted replicas."],
    },
    "high_latency": {
        "high":   ["Profile DataNode disk I/O at write time — check for saturation or slow disks.",
                   "Check network bandwidth between NameNode and DataNodes during latency window."],
        "medium": ["Review JVM GC logs on affected DataNode for stop-the-world pauses.",
                   "Verify DataNode CPU is not contended by co-located processes."],
        "low":    ["Move high-throughput workloads to dedicated DataNodes to reduce latency variance."],
    },
}


# ── FIX 1: Scan entire doc for E-codes, not just Event Sequence line ──────────
def build_event_context_for_prompt(doc: str) -> str:
    """Old code used single-line regex — missed E-codes when sequence spanned lines."""
    ids = list(dict.fromkeys(re.findall(r"\bE\d+\b", doc)))
    if not ids:
        return "No events found."
    return "\n".join(f"{e}: {EVENT_DESCRIPTIONS.get(e, 'Unknown event')}" for e in ids)


# ── FIX 2: system prompt explicitly forbids placeholder text ──────────────────
SYSTEM_PROMPT = """You are an expert in HDFS log analysis and anomaly diagnosis.
Return strictly valid JSON only.
Never use placeholder text like "...", "urgent action 1", "follow-up 1", or empty arrays.
Every field must contain real, specific analysis based on the block data provided."""


# ── compress_historical_case (unchanged) ──────────────────────────────────────
def compress_historical_case(doc: str, block_id: str, latency: int, distance: float) -> str:
    seq_m   = re.search(r"Event Sequence\s*:\s*(.+)", doc)
    atype_m = re.search(r"Anomaly Type\(s\)\s*:\s*(.+)", doc)
    total_m = re.search(r"Total Events\s*:\s*(\d+)", doc)
    seq     = seq_m.group(1).strip()   if seq_m   else ""
    atype   = atype_m.group(1).strip() if atype_m else "unknown"
    total   = total_m.group(1)         if total_m else "?"
    tokens  = [t.strip() for t in seq.split("->") if t.strip()]
    short   = " -> ".join(tokens[:15])
    if len(tokens) > 15:
        short += f" (+{len(tokens)-15} more)"
    return (f"Block: {block_id} | {atype} | {latency}ms | dist={distance:.3f} | events={total}\n"
            f"Seq: {short}")


# ── FIX 3: prompt uses angle-bracket instructions, not placeholder values ──────
def build_user_prompt(query_doc: str, historical_context: str, query_meta: dict) -> str:
    """Old prompt had \"summary\": \"...\" which the LLM copied literally."""
    block_id     = query_meta["block_id"]
    anomaly_type = query_meta.get("anomaly_types", query_meta.get("primary_type", "unknown"))
    latency_ms   = query_meta.get("latency", 0)
    total_events = query_meta.get("total_events", 0)
    event_ref    = build_event_context_for_prompt(query_doc)

    seq_m     = re.search(r"Event Sequence\s*:\s*(.+?)(?:\n|$)", query_doc)
    event_seq = seq_m.group(1).strip() if seq_m else " -> ".join(re.findall(r"\bE\d+\b", query_doc))

    return f"""You are diagnosing an HDFS block anomaly. Return ONLY a JSON object — no markdown, no explanation.

BLOCK UNDER ANALYSIS:
- Block ID: {block_id}
- Anomaly Type: {anomaly_type}
- Total Events: {total_events}
- Latency: {latency_ms}ms
- Event Sequence: {event_seq}

EVENT DESCRIPTIONS (use these verbatim in event_explanations):
{event_ref}

SIMILAR HISTORICAL CASES:
{historical_context if historical_context else "No similar cases found."}

Return this JSON with real analysis in every field — no placeholders, no empty arrays:
{{
  "block_id": "{block_id}",
  "anomaly_type": "<one of: duplicate_pattern | repetition | missing_events | high_latency>",
  "summary": "<2 sentences; must mention {total_events} events and {latency_ms}ms latency>",
  "root_cause": "<explain failure mechanism; cite specific event IDs like E5, E11 as evidence>",
  "comparison_to_historical": "<one sentence comparing to historical cases; include a specific latency number>",
  "mitigation_steps": {{
    "high":   ["<urgent action specific to this anomaly>", "<urgent action 2>"],
    "medium": ["<follow-up action>", "<follow-up action 2>"],
    "low":    ["<long-term improvement>"]
  }},
  "event_explanations": {{
    "<event ID>": "<description from EVENT DESCRIPTIONS — one entry per unique event in the sequence>"
  }}
}}"""


# ── FIX 4: compute_confidence guards against division by zero ─────────────────
def compute_confidence(similar_docs: list, query_doc: str, latency: int) -> tuple:
    avg_dist = sum(d["distance"] for d in similar_docs) / max(len(similar_docs), 1)
    retrieval_score = 0.9 if avg_dist < 0.70 else (0.7 if avg_dist < 0.80 else 0.5)

    event_ids = re.findall(r"\bE\d+\b", query_doc)
    if event_ids:  # FIX: guard division by zero when doc has no events
        counts = Counter(event_ids)
        repetition_score = sum(v for v in counts.values() if v > 1) / len(event_ids)
    else:
        repetition_score = 0.0

    latency_score = 0.9 if latency > 20000 else (0.7 if latency > 8000 else 0.5)

    confidence = round(min(max((retrieval_score + repetition_score + latency_score) / 3, 0.0), 1.0), 2)
    label = "high" if confidence >= 0.75 else ("medium" if confidence >= 0.4 else "low")
    return confidence, label


# ── JSON repair (standalone — not nested inside generate_root_cause) ──────────
def repair_json(s: str) -> dict:
    s = s.strip()
    s = re.sub(r"^```(?:json)?\s*", "", s)
    s = re.sub(r"\s*```$", "", s)
    s = re.sub(r",\s*([}\]])", r"\1", s)
    s = re.sub(r"(?<![\w])'([^']*)'(?![\w])", r'"\1"', s)
    result = []; in_string = False; i = 0
    while i < len(s):
        c = s[i]
        if c == "\\" and in_string:
            result.append(c); i += 1
            if i < len(s): result.append(s[i])
            i += 1; continue
        if c == '"':
            if not in_string:
                in_string = True; result.append(c)
            else:
                rest   = s[i+1:].lstrip()
                closes = (not rest) or rest[0] in (":", ",", "}", "]")
                if closes or i == len(s) - 1:
                    in_string = False; result.append(c)
                else:
                    result.append('\\"')
            i += 1; continue
        result.append(c); i += 1
    s = "".join(result)
    s = re.sub(r",\s*([}\]])", r"\1", s)
    try:
        return json.loads(s)
    except json.JSONDecodeError:
        if in_string: s += '"' 
        depth = []
        for ch in s:
            if ch in ("{", "["): depth.append("}" if ch == "{" else "]")
            elif ch in ("}", "]") and depth: depth.pop()
        s += "".join(reversed(depth))
        s = re.sub(r",\s*([}\]])", r"\1", s)
        return json.loads(s)


# ── FIX 5: generate_root_cause — clean structure, repair_json runs correctly ──
# Old code had repair_json defined INSIDE the try block, and
# 'parsed = repair_json(content)' indented INTO repair_json's body — it never ran.
def generate_root_cause(query_doc: str, historical_context: str, query_meta: dict, similar_docs: list) -> dict:
    user_prompt = build_user_prompt(query_doc, historical_context, query_meta)

    try:
        response = groq_client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": user_prompt},
            ],
            temperature=0.2,
            max_tokens=1500,
        )
    except Exception as e:
        return {"error": f"Groq API call failed: {str(e)}", "block_id": query_meta.get("block_id", "unknown")}

    content = response.choices[0].message.content.strip()

    try:
        parsed = repair_json(content)
    except Exception as parse_err:
        print(f"[PARSE ERROR] block={query_meta.get('block_id','?')} | {parse_err}")
        print(f"  Raw (first 300 chars): {content[:300]}")
        parsed = {
            "block_id":                  query_meta.get("block_id", "parse_error"),
            "summary":                   "JSON parse failed — see printed error above.",
            "root_cause":                content[:500],
            "comparison_to_historical":  "",
            "event_explanations":        {},
            "anomaly_type":              query_meta.get("primary_type", "unknown"),
            "mitigation_steps":          {"high": [], "medium": [], "low": []},
            "parse_error":               True,
        }

    # Enforce valid anomaly type
    if parsed.get("anomaly_type") not in VALID_ANOMALY_TYPES:
        parsed["anomaly_type"] = query_meta.get("primary_type", "unknown")

    # Fill empty mitigation tiers with typed fallbacks
    atype    = parsed.get("anomaly_type", query_meta.get("primary_type", "duplicate_pattern"))
    fallback = FALLBACK_MITIGATIONS.get(atype, FALLBACK_MITIGATIONS["duplicate_pattern"])
    mit      = parsed.get("mitigation_steps")
    if not isinstance(mit, dict):
        parsed["mitigation_steps"] = fallback
    else:
        for tier in ("high", "medium", "low"):
            cleaned = [x for x in (mit.get(tier) or []) if x and str(x).strip()]
            mit[tier] = cleaned if cleaned else fallback[tier]

    # Fix comparison_to_historical if it came back as a list/JSON fragment
    cth = parsed.get("comparison_to_historical", "")
    if isinstance(cth, (list, dict)) or (isinstance(cth, str) and cth.strip().startswith("[")):
        avg = int(sum(r["latency"] for r in similar_docs) / max(len(similar_docs), 1))
        parsed["comparison_to_historical"] = (
            f"Historical cases averaged {avg}ms latency vs this block's {query_meta.get('latency',0)}ms."
        )

    # Override confidence with deterministic score
    confidence, label = compute_confidence(similar_docs, query_doc, query_meta.get("latency", 0))
    parsed["confidence"]       = confidence
    parsed["confidence_label"] = label

    return parsed


## Step 7: Query Constructor

In [ ]:
# ── Full auto-query: shows complete event sequence with no truncation ─────

def build_query_from_doc(doc: str, meta: dict) -> str:
    """
    Build the ChromaDB retrieval query from the block's parsed metadata.
    The full event sequence is included — no character-limit truncation.
    """
    anomaly_types = meta.get('anomaly_types', 'unknown')
    latency       = meta.get('latency', 0)
    total_events  = meta.get('total_events', 0)

    seq_match = re.search(r'Event Sequence\s*:\s*(.+)', doc)
    event_seq = seq_match.group(1).strip() if seq_match else ''

    query = (
        f"HDFS block anomaly: {anomaly_types}. "
        f"Latency: {latency}ms. "
        f"Total events: {total_events}. "
        f"Event sequence: {event_seq}"
    )
    return query


# Demo
sample_query = build_query_from_doc(documents[0], all_metadata[0])
print('Auto-constructed query (full, no truncation):')
print(sample_query)


## Step 8: Single Block — End-to-End Pipeline Test

Diagnose a single block to verify the whole pipeline works correctly before batch mode.
You can change `TEST_INDEX` to try a different block.

In [ ]:
TEST_INDEX = 0   # change to test a different block

query_doc  = documents[TEST_INDEX]
query_meta = all_metadata[TEST_INDEX]

print(f"Diagnosing block : {query_meta['block_id']}")
print(f"Anomaly type     : {query_meta['anomaly_types']}")
print(f"Latency          : {query_meta['latency']} ms")
print()

auto_query = build_query_from_doc(query_doc, query_meta)
print("Auto-query (full):"); print(auto_query)
print()

similar_docs = retrieve_similar_anomalies(
    query               = auto_query,
    k                   = 3,
    anomaly_type_filter = query_meta["primary_type"],
)

print("Retrieved similar cases:")
for i, r in enumerate(similar_docs):
    print(f"  Case {i+1}: block_id={r['block_id']}, "
          f"type={r['primary_type']}, latency={r['latency']}ms, distance={r['distance']:.4f}")
print()

# FIX: Use compress_historical_case instead of full documents.
# Passing full documents into historical_context causes 413 token-limit errors.
historical_context = "\n\n".join([
    f"Case {i+1}\n" +
    compress_historical_case(r["document"], r["block_id"], r["latency"], r["distance"])
    for i, r in enumerate(similar_docs)
    if r["block_id"] != query_meta["block_id"]
])

result = generate_root_cause(query_doc, historical_context, query_meta, similar_docs)

print("LLM Root Cause Analysis:")
print(json.dumps(result, indent=2))


## Step 9: Frontend Query Examples

These are the query patterns your frontend should send to `POST /analyze`.
Three modes are supported: by `block_id`, by free-text `query`, and with an optional `anomaly_type_filter`.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# FRONTEND QUERY EXAMPLES
# These are example payloads your frontend sends to POST /analyze
# Copy any of these into Postman / your JS fetch / your React form
# ─────────────────────────────────────────────────────────────────────────────

import json

EXAMPLE_QUERIES = [
    {
        "_description": "Mode 1: Diagnose a specific block by its ID (most common use case)",
        "payload": {
            "block_id": "blk_8462687553742484299"
        }
    },
    {
        "_description": "Mode 2: Diagnose by block ID and restrict retrieved evidence to same anomaly type",
        "payload": {
            "block_id": "blk_8462687553742484299",
            "anomaly_type_filter": "duplicate_pattern"
        }
    },
    {
        "_description": "Mode 3: Free-text query — useful when frontend user describes symptoms manually",
        "payload": {
            "query": "HDFS block with repeated E5 and E11 events, high latency around 7000ms, duplicate pipeline writes"
        }
    },
    {
        "_description": "Mode 4: Free-text query filtered to a specific anomaly type",
        "payload": {
            "query": "Block replication keeps retrying, DataNode appears stuck in a loop",
            "anomaly_type_filter": "repetition"
        }
    },
    {
        "_description": "Mode 5: Missing events — pipeline aborted before completion",
        "payload": {
            "query": "Block write pipeline aborted mid-execution, fewer than 5 events recorded, block may be corrupt",
            "anomaly_type_filter": "missing_events"
        }
    },
    {
        "_description": "Mode 6: High latency — block completed but took too long",
        "payload": {
            "query": "Block pipeline completed successfully but latency was over 30000ms, possible disk or network bottleneck",
            "anomaly_type_filter": "high_latency"
        }
    },
    {
        "_description": "Mode 7: Retrieve more historical cases (k=5) for a deeper analysis",
        "payload": {
            "block_id": "blk_8462687553742484299",
            "k": 5
        }
    },
]

print("=" * 65)
print("FRONTEND QUERY EXAMPLES — send these as POST /analyze body")
print("=" * 65)
for ex in EXAMPLE_QUERIES:
    print(f"\n{ex['_description']}")
    print(json.dumps(ex['payload'], indent=2))

print()
print("=" * 65)
print("JS fetch example (paste into your frontend):")
print("=" * 65)
print("""
const response = await fetch('YOUR_CLOUDFLARE_URL/analyze', {
  method: 'POST',
  headers: { 'Content-Type': 'application/json' },
  body: JSON.stringify({
    block_id: 'blk_8462687553742484299'   // or use query: 'your description'
  })
});
const result = await response.json();
console.log(result);
""")


## Step 10: Batch Analysis

Run the full pipeline over a sample of blocks and save results to CSV.
Adjust `SAMPLE_SIZE` and `SAMPLE_PER_TYPE` as needed.

In [ ]:
import time

# ── Balanced sample: pick N blocks per anomaly type ───────────────────────────
SAMPLE_PER_TYPE = 5   # blocks to diagnose per anomaly type
SLEEP_BETWEEN   = 3.0 # seconds between Groq calls (rate-limit safety)

meta_df = pd.DataFrame(all_metadata)
meta_df["doc_index"] = meta_df.index

sampled_indices = (
    meta_df
    .groupby("primary_type", group_keys=False)
    .apply(lambda g: g.sample(min(SAMPLE_PER_TYPE, len(g)), random_state=42), include_groups=False)
    ["doc_index"]
    .tolist()
)

print(f"Blocks to diagnose: {len(sampled_indices)}")

batch_results = []

for idx in sampled_indices:
    q_doc  = documents[idx]
    q_meta = all_metadata[idx]

    print(f"[{idx}] {q_meta['block_id']} | {q_meta['primary_type']}")

    auto_query = build_query_from_doc(q_doc, q_meta)

    similar = retrieve_similar_anomalies(
        query               = auto_query,
        k                   = 3,
        anomaly_type_filter = q_meta["primary_type"],
    )

    hist_ctx = "\n\n".join([
        f"Case {i+1}\n" +
        compress_historical_case(r['document'], r['block_id'], r['latency'], r['distance'])
        for i, r in enumerate(similar)
        if r["block_id"] != q_meta["block_id"]
    ])

    #llm_result = generate_root_cause(q_doc, hist_ctx)
    llm_result = generate_root_cause(q_doc, hist_ctx, q_meta, similar)

    # Flatten for CSV export
    batch_results.append({
        "block_id":               q_meta["block_id"],
        "primary_type":           q_meta["primary_type"],
        "latency_ms":             q_meta["latency"],
        "llm_block_id":           llm_result.get("block_id", ""),
        "summary":                llm_result.get("summary", ""),
        "root_cause":             llm_result.get("root_cause", ""),
        "comparison_to_historical": llm_result.get("comparison_to_historical", ""),
        "anomaly_type_llm":       llm_result.get("anomaly_type", ""),
        "confidence":             llm_result.get("confidence", 0.0),
        "confidence_label":       llm_result.get("confidence_label", ""),
        "mitigation_high":        " | ".join(s for s in llm_result.get("mitigation_steps", {}).get("high", [])   or [] if s),
        "mitigation_medium":      " | ".join(s for s in llm_result.get("mitigation_steps", {}).get("medium", []) or [] if s),
        "mitigation_low":         " | ".join(s for s in llm_result.get("mitigation_steps", {}).get("low", [])    or [] if s),
        "error":                  llm_result.get("error", ""),
    })

    time.sleep(SLEEP_BETWEEN)

batch_df = pd.DataFrame(batch_results)

out_path = "/content/drive/MyDrive/Log_Anamoly_Detection/results/llm_root_cause_results.csv"
batch_df.to_csv(out_path, index=False)

print(f"\nSaved {len(batch_df)} results to {out_path}")
print(batch_df[["block_id", "primary_type", "confidence_label", "summary"]].to_string())